# HEATWIN Phase 4 — discrete PID, nonlinear closed-loop validation

All experiments use the *existing* Phase 2 nonlinear plant and the *saved* Phase 3 FOPDT model. No twin or fault diagnostics are implemented.

In [ ]:
from pathlib import Path

from heatwin.config import load_config
from heatwin.control.tuning import imc_pi, load_identified_model, ziegler_nichols
from heatwin.diagnostics.control_metrics import evaluate_control
from heatwin.simulation.closed_loop import SetpointStep, simulate_closed_loop
model=load_identified_model(Path("configs/identified/phase3_fopdt_nominal.json"))
p=load_config()
print(model)

In [ ]:
settings=imc_pi(model, lambda_s=model.tau_s)
hold=simulate_closed_loop(p, settings, duration_s=300)
assert abs(hold.tc_c.iloc[-1]-p.setpoint)<1e-6
print("Nominal equilibrium valve command:",hold.u.iloc[0])

In [ ]:
change=[SetpointStep(60,57)]
pi=simulate_closed_loop(p, settings, duration_s=420, setpoint_steps=change)
pid=simulate_closed_loop(p, ziegler_nichols(model, mode="PID"), duration_s=420, setpoint_steps=change)
for name,data in (("IMC PI",pi),("Z-N PID",pid)):
    print(name, evaluate_control(data,window_start_s=60,reference_initial_c=55))

In [ ]:
import matplotlib.pyplot as plt

fig,ax=plt.subplots(figsize=(9,4))
ax.plot(pi.time_s,pi.tc_c,label="IMC-PI")
ax.plot(pid.time_s,pid.tc_c,label="ZN-PID")
ax.step(pi.time_s,pi.setpoint_c,where="post",label="setpoint",linestyle="--")
ax.set(xlabel="Time (s)",ylabel="Cold outlet (°C)",title="Real closed-loop nonlinear heat exchanger response")
ax.legend()
ax.grid(alpha=0.3)
plt.show()